[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/editorial-v2/notebooks/04_Vertical_Forces_and_Motion.ipynb)

# DiveLab

## Notebook 04 — Vertical Forces and Motion

**Guiding question:** How does a buoyancy imbalance change a diver's vertical motion?

*Companion laboratory for Chapter 4 — Buoyancy and Vertical Dynamics.*


## Learning objectives

By the end of this lab, you will be able to:

- apply the DiveLab depth, velocity and force sign conventions;
- calculate initial vertical acceleration from a buoyancy imbalance;
- implement a signed quadratic-drag model;
- determine terminal ascent or descent velocity;
- connect Boyle's law and Archimedes' principle to depth-dependent net force;
- identify the reinforcing depth–buoyancy mechanism;
- prepare the force model for state-space representation and later numerical integration.


## From Notebook 03 to Notebook 04

Notebook 03 completed the static chain

$$
z
\longrightarrow
P_{\mathrm{abs}}(z)
\longrightarrow
V_g(z)
\longrightarrow
F_B(z).
$$

We now add Newton's second law:

$$
F_{\mathrm{net}}
\longrightarrow
\text{acceleration}
\longrightarrow
\text{velocity}
\longrightarrow
\text{depth}.
$$

This closes the first feedback mechanism in DiveLab. We evaluate the differential equations here without yet introducing state-space notation or numerical time-stepping.


## Model conventions and assumptions

We use the following transparent teaching model:

- depth $z\geq0$ is positive downward;
- vertical velocity $v$ is positive upward, so $\dot z=-v$;
- scalar forces are positive upward;
- mass $m$ is constant;
- water density $\rho$ and gravity $g$ are constant;
- flexible gas is ideal and isothermal;
- drag is represented by one quadratic coefficient;
- waves, currents, added mass, breathing cycles and diver control actions are omitted.

The model is educational and must not be used to select operational ascent or descent rates.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## Physical and model parameters

The drag parameter is

$$
c=\frac{1}{2}\rho C_DA.
$$

Its value combines fluid density, body orientation, projected area and an empirical drag coefficient.


In [ ]:
g = 9.80665                 # gravitational acceleration [m/s^2]
rho = 1025.0                # representative seawater density [kg/m^3]
P0 = 101_325.0              # surface pressure [Pa]

mass_kg = 85.0              # diver + equipment mass [kg]
drag_coefficient = 0.80     # dimensionless
projected_area_m2 = 0.70    # representative projected area [m^2]

drag_parameter = 0.5 * rho * drag_coefficient * projected_area_m2

print(f"Drag parameter c = {drag_parameter:.1f} kg/m")


## Signed vertical force model

With upward velocity positive, quadratic drag is

$$
F_D(v)=-c\,v|v|.
$$

Newton's second law becomes

$$
m\dot v=\Delta F_B-c\,v|v|,
$$

where $\Delta F_B=F_B-mg$ is the buoyancy imbalance.


In [ ]:
def drag_force_upward(velocity_m_s, drag_parameter_kg_m=drag_parameter):
    """Return signed vertical drag [N], positive upward."""
    velocity_m_s = np.asarray(velocity_m_s)
    return -drag_parameter_kg_m * velocity_m_s * np.abs(velocity_m_s)


def vertical_acceleration(
    buoyancy_imbalance_n,
    velocity_m_s,
    mass_kg=mass_kg,
    drag_parameter_kg_m=drag_parameter,
):
    """Return upward acceleration [m/s^2]."""
    drag_n = drag_force_upward(velocity_m_s, drag_parameter_kg_m)
    return (buoyancy_imbalance_n + drag_n) / mass_kg


## Check the drag direction

During ascent ($v>0$), drag must be downward. During descent ($v<0$), it must be upward. At rest, drag is zero.


In [ ]:
for velocity_m_s in [-0.30, 0.0, 0.30]:
    drag_n = float(drag_force_upward(velocity_m_s))
    print(
        f"v = {velocity_m_s:+.2f} m/s"
        f"  ->  F_D = {drag_n:+.2f} N"
    )


## Initial acceleration

At $v=0$, drag vanishes:

$$
\dot v(0)=\frac{\Delta F_B}{m}.
$$

Compare negative, neutral and positive buoyancy for the same mass.


In [ ]:
imbalances_n = np.array([-12.0, 0.0, 12.0])

print("Imbalance [N]   Initial acceleration [m/s^2]")
for imbalance_n in imbalances_n:
    acceleration_m_s2 = vertical_acceleration(imbalance_n, 0.0)
    print(f"{imbalance_n:13.1f}   {acceleration_m_s2:27.3f}")


## Terminal velocity

For a constant buoyancy imbalance,

$$
v_\infty
=\operatorname{sgn}(\Delta F_B)
\sqrt{\frac{|\Delta F_B|}{c}}.
$$


In [ ]:
def terminal_velocity(
    buoyancy_imbalance_n,
    drag_parameter_kg_m=drag_parameter,
):
    """Return signed terminal vertical velocity [m/s]."""
    imbalance = np.asarray(buoyancy_imbalance_n, dtype=float)
    return (
        np.sign(imbalance)
        * np.sqrt(np.abs(imbalance) / drag_parameter_kg_m)
    )


for imbalance_n in [-15.0, -8.0, 0.0, 8.0, 15.0]:
    terminal_m_s = float(terminal_velocity(imbalance_n))
    print(
        f"Delta F_B = {imbalance_n:+5.1f} N"
        f"  ->  v_terminal = {terminal_m_s:+.3f} m/s"
    )


## Acceleration versus velocity

Terminal velocity is the zero crossing of the acceleration curve. Drag drives the acceleration toward zero as speed increases.


In [ ]:
velocity_grid_m_s = np.linspace(-0.40, 0.40, 401)

plt.figure(figsize=(8, 5))

for imbalance_n in [-10.0, 0.0, 10.0]:
    acceleration_m_s2 = vertical_acceleration(
        imbalance_n,
        velocity_grid_m_s,
    )
    plt.plot(
        velocity_grid_m_s,
        acceleration_m_s2,
        label=fr"$\Delta F_B={imbalance_n:+.0f}$ N",
    )

plt.axhline(0.0, color="black", linewidth=0.8)
plt.axvline(0.0, color="black", linewidth=0.8)
plt.xlabel("Upward velocity [m/s]")
plt.ylabel("Upward acceleration [m/s²]")
plt.title("Quadratic drag limits vertical velocity")
plt.grid(True)
plt.legend()
plt.show()


## Interpretation

For positive buoyancy, acceleration is initially upward and falls to zero at a positive terminal velocity. For negative buoyancy, the zero crossing occurs at a negative velocity. Under neutral buoyancy, drag opposes any existing motion and the only zero-acceleration point is $v=0$.

Neutral buoyancy is therefore not the same as zero velocity.


## Reuse Boyle and Archimedes

For a fixed displaced volume $V_f$ and a flexible surface-gas volume $V_{g0}$,

$$
V_g(z)=V_{g0}\frac{P_0}{P_0+\rho gz},
$$

and

$$
F_B(z)=\rho g\left[V_f+V_g(z)\right].
$$


In [ ]:
def pressure_at_depth(depth_m):
    """Return absolute ambient pressure [Pa]."""
    return P0 + rho * g * np.asarray(depth_m)


def gas_volume_at_depth(depth_m, surface_gas_volume_m3):
    """Return ideal flexible-gas volume [m^3]."""
    return surface_gas_volume_m3 * P0 / pressure_at_depth(depth_m)


def buoyant_force_at_depth(
    depth_m,
    fixed_volume_m3,
    surface_gas_volume_m3,
):
    """Return total upward buoyant force [N]."""
    gas_volume_m3 = gas_volume_at_depth(
        depth_m,
        surface_gas_volume_m3,
    )
    return rho * g * (fixed_volume_m3 + gas_volume_m3)


def buoyancy_imbalance_at_depth(
    depth_m,
    fixed_volume_m3,
    surface_gas_volume_m3,
    mass_kg=mass_kg,
):
    """Return buoyancy minus weight [N]."""
    return (
        buoyant_force_at_depth(
            depth_m,
            fixed_volume_m3,
            surface_gas_volume_m3,
        )
        - mass_kg * g
    )


## Construct a neutral operating depth

Choose $z^\ast=20\ \mathrm{m}$ and a flexible surface-gas volume of $8\ \mathrm{L}$. The required fixed displaced volume is

$$
V_f=\frac{m}{\rho}-V_g(z^\ast).
$$


In [ ]:
neutral_depth_m = 20.0
surface_gas_volume_l = 8.0
surface_gas_volume_m3 = surface_gas_volume_l / 1000

gas_volume_at_neutral_m3 = gas_volume_at_depth(
    neutral_depth_m,
    surface_gas_volume_m3,
)
fixed_volume_m3 = mass_kg / rho - gas_volume_at_neutral_m3

print(f"Target neutral depth:       {neutral_depth_m:.1f} m")
print(f"Gas volume at that depth:   {gas_volume_at_neutral_m3*1000:.3f} L")
print(f"Required fixed volume:      {fixed_volume_m3*1000:.3f} L")
print(
    "Net force at target depth: "
    f"{buoyancy_imbalance_at_depth(neutral_depth_m, fixed_volume_m3, surface_gas_volume_m3):+.3e} N"
)


## Net force around the neutral depth

At zero velocity, drag is zero. The remaining net force shows how a small depth displacement changes acceleration.


In [ ]:
depth_grid_m = np.linspace(10.0, 30.0, 301)
imbalance_grid_n = buoyancy_imbalance_at_depth(
    depth_grid_m,
    fixed_volume_m3,
    surface_gas_volume_m3,
)
acceleration_at_rest_m_s2 = imbalance_grid_n / mass_kg

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(depth_grid_m, imbalance_grid_n)
axes[0].axhline(0.0, color="black", linewidth=0.8)
axes[0].axvline(neutral_depth_m, color="black", linestyle="--")
axes[0].set_xlabel("Depth [m]")
axes[0].set_ylabel("Buoyancy imbalance [N]")
axes[0].set_title("Net force changes with depth")
axes[0].grid(True)

axes[1].plot(depth_grid_m, acceleration_at_rest_m_s2)
axes[1].axhline(0.0, color="black", linewidth=0.8)
axes[1].axvline(neutral_depth_m, color="black", linestyle="--")
axes[1].set_xlabel("Depth [m]")
axes[1].set_ylabel("Upward acceleration at rest [m/s²]")
axes[1].set_title("Acceleration follows the force imbalance")
axes[1].grid(True)

plt.tight_layout()
plt.show()


## What the plots show

At the chosen neutral depth, buoyancy equals weight. A small ascent decreases depth, expands the gas and produces positive net upward force. A small descent compresses the gas and produces negative net upward force.

The physical chain is reinforcing:

$$
\text{rise}
\longrightarrow
\text{gas expansion}
\longrightarrow
\text{greater buoyancy}
\longrightarrow
\text{upward acceleration}.
$$

We can diagnose this mechanism without yet integrating the equations through time.


## Evaluate the complete acceleration model

At any chosen depth and velocity,

$$
\dot v
=\frac{F_B(z)-mg-c\,v|v|}{m}.
$$


In [ ]:
def coupled_vertical_acceleration(depth_m, velocity_m_s):
    """Return upward acceleration for the coupled model [m/s^2]."""
    imbalance_n = buoyancy_imbalance_at_depth(
        depth_m,
        fixed_volume_m3,
        surface_gas_volume_m3,
    )
    return vertical_acceleration(imbalance_n, velocity_m_s)


test_states = [
    (19.0, 0.0),
    (20.0, 0.0),
    (21.0, 0.0),
    (20.0, 0.20),
    (20.0, -0.20),
]

print("Depth [m]   Velocity [m/s]   Acceleration [m/s^2]")
for depth_m, velocity_m_s in test_states:
    acceleration_m_s2 = coupled_vertical_acceleration(
        depth_m,
        velocity_m_s,
    )
    print(
        f"{depth_m:9.1f}   {velocity_m_s:14.2f}"
        f"   {acceleration_m_s2:21.4f}"
    )


## First dynamical model

The complete equations are

$$
\begin{aligned}
\dot z
&=-v,\\
\dot v
&=
\frac{
\rho g\left[
V_f+V_{g0}\dfrac{P_0}{P_0+\rho gz}
\right]
-mg
-c\,v|v|
}{m}.
\end{aligned}
$$

Depth and vertical velocity are the minimum time-varying quantities required to predict the model's future motion. Chapter 5 will formalize them as a state vector.


## Scope and model limitations

This model omits active BCD operation, breathing, exposure-suit material behavior, gas migration, body orientation, added mass, waves, currents and diver control. The drag parameters are aggregate teaching values. The results are not operational dive guidance.


## Exercises

### 1. Initial acceleration

Calculate the initial acceleration of a $90\ \mathrm{kg}$ system with buoyancy imbalances of $-15$, $0$, and $+15\ \mathrm{N}$.


In [ ]:
# Your code here


### 2. Terminal velocity

Plot terminal velocity versus buoyancy imbalance from $-20$ to $+20\ \mathrm{N}$. Explain the curve's sign and shape.


In [ ]:
# Your code here


### 3. Change the drag model

Repeat the acceleration-versus-velocity plot for two projected areas. Explain how area changes terminal velocity.


In [ ]:
# Your code here


### 4. Move the neutral operating depth

Choose a target neutral depth of $15\ \mathrm{m}$. Recalculate $V_f$, then plot net force from $5$ to $25\ \mathrm{m}$.


In [ ]:
# Your code here


## Challenge — prepare for time integration

Write a function named vertical_rhs that accepts depth and velocity and returns the pair $[\dot z,\dot v]$.

Evaluate it at several depths and velocities. Do not integrate it yet; a later computational lab will show how a time-stepping algorithm turns these derivatives into a trajectory.


In [ ]:
# Your code here


## Summary

In this lab we learned that:

- the DiveLab sign convention gives $\dot z=-v$;
- Newton's second law converts net force into acceleration;
- signed quadratic drag always opposes motion;
- constant buoyancy imbalance produces a terminal velocity;
- neutral buoyancy does not imply zero velocity;
- Boyle's law makes flexible-gas buoyancy depend on depth;
- the depth–buoyancy coupling is reinforcing;
- the equations are ready for state-space representation and time integration.

### Core insight

> Force changes motion, and motion changes the environment that generates the force.


## Next notebook

The next companion notebook will formalize depth and vertical velocity as a state vector. Numerical time integration will follow after that model structure is established.
